# Hands-on 27 (Bab 11): Ekspor Model Sederhana ke Format Ringan

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 11.

Dua jalur deployment edge yang realistis:
- **Jalur A:** Random Forest scikit-learn -> joblib (untuk Raspberry Pi dengan Python).
- **Jalur B:** MLP kecil Keras -> .keras -> TensorFlow Lite (untuk inferensi edge).

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Jalur A: Ekspor scikit-learn (joblib)

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/export_sklearn_model.py"], capture_output=True, text=True)
print(r.stdout)

Model scikit-learn berhasil disimpan ke: models/room_status_rf.joblib



## 2. Jalur B: Keras -> TFLite

In [3]:
r = subprocess.run([sys.executable, "scripts/export_keras_to_tflite.py"], capture_output=True, text=True)
print(r.stdout)

Epoch 1/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 22s 482ms/step - accuracy: 0.9688 - loss: 2.7495
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 773us/step - accuracy: 0.9589 - loss: 0.5321 
Epoch 2/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - accuracy: 1.0000 - loss: 0.0017
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 711us/step - accuracy: 0.9788 - loss: 0.1730
Epoch 3/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 1.0000 - loss: 6.8533e-07
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 706us/step - accuracy: 0.9808 - loss: 0.2255   
Epoch 4/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9688 - loss: 0.3645
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 727us/step - accuracy: 0.9795 - loss: 0.1747
Epoch 5/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/s

## 3. Verifikasi Inferensi dari File Model

In [4]:
import joblib
import numpy as np
import pandas as pd

# Inferensi ulang dari model joblib
model = joblib.load("models/room_status_rf.joblib")
X_test = pd.read_csv("data/processed/classification_status/X_test.csv")
print("[INFO] Prediksi joblib (5 baris):", list(model.predict(X_test.head())))

# Inferensi ulang dari model TFLite (batch dim default 1: resize dulu ke 5 baris)
import tensorflow as tf
interpreter = tf.lite.Interpreter(model_path="models/room_binary.tflite")
Xb = pd.read_csv("data/processed/classification_room/X_test.csv").values.astype("float32")[:5]
inp_detail = interpreter.get_input_details()[0]
interpreter.resize_tensor_input(inp_detail["index"], Xb.shape)
interpreter.allocate_tensors()
interpreter.set_tensor(inp_detail["index"], Xb)
interpreter.invoke()
out = interpreter.get_output_details()[0]["index"]
print("[INFO] Probabilitas TFLite (5 baris):", interpreter.get_tensor(out).flatten().round(3).tolist())

[INFO] Prediksi joblib (5 baris): ['normal', 'normal', 'normal', 'normal', 'normal']


I0000 00:00:1792650323.936255   11789 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792650323.959721   11789 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1792650325.040981   11789 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[INFO] Probabilitas TFLite (5 baris): [0.5429999828338623, 0.0, 0.0, 0.0, 0.0]


/home/sandbox/.local/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## Penutup

Training dan deployment adalah dua tahap berbeda. Model joblib (sekitar 425 KB) siap
untuk Raspberry Pi dengan Python biasa; model TFLite hanya sekitar 3.9 KB dan bisa
berjalan dengan interpreter ringan tanpa TensorFlow penuh. Keduanya terverifikasi bisa
dimuat ulang dan dipakai inferensi dari file hasil ekspor di `models/`.

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 29 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 7. Apa yang harus dipahami pembaca?

Pada tahap ini pembaca belajar bahwa:

model training bisa tetap dilakukan di laptop,
model lalu disimpan ke format yang cocok untuk deployment,
joblib cukup realistis untuk model klasik di Raspberry Pi,
TensorFlow Lite adalah jalur praktis untuk model neural network ringan.

Ini konsisten dengan arsitektur edge modern, di mana edge dipakai untuk inferensi
dan cloud/laptop dipakai untuk training dan eksperimen.